In [42]:
from tqdm.auto import tqdm
import pickle
from functools import partial
import numpy as np
import torch
import os
import time
from collections import defaultdict
import torch
from joblib import Parallel, delayed
from joblib.externals.loky import get_reusable_executor


In [2]:
def compute_tpir_from_heap(neg_heap, pos_scores, total_neg_pairs, target_fars):
    """
    从堆中计算 TPIR
    """
    # 将堆转换为排序数组（从高到低）
    neg_scores_sorted = sorted(neg_heap, reverse=True)
    
    results = {}
    thresholds = {}
    
    for far in target_fars:
        # 计算对应的索引
        idx = int(far * total_neg_pairs)
        
        if idx < len(neg_scores_sorted):
            threshold = neg_scores_sorted[idx]
        else:
            # 如果 FAR 太小，使用最小的负样本分数
            threshold = neg_scores_sorted[-1] if neg_scores_sorted else 0.0
        
        # 计算 TPIR
        tpir = np.mean(pos_scores >= threshold)if len(pos_scores) > 0 else 0.0
        
        results[f'tpir_at_far_{far}'] = float(tpir) * 100 
        thresholds[far] = threshold
    
    return results, thresholds
def compute_tpir_optimized(query_feats_list, query_ids, target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]):
    """
    优化版本：只维护 top 1e-5 的负样本分数
    """
    
    N = len(query_ids)
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    feats_t = torch.tensor(query_feats_list, dtype=torch.float32).to(device)
    ids_t = torch.tensor(query_ids).to(device)

    # 预计算总的负样本数量（基于上三角矩阵）
    total_pairs = N * (N - 1) // 2  # 上三角对数
    unique_ids, counts = np.unique(query_ids, return_counts=True)
    total_pos_pairs = sum(c * (c - 1) // 2 for c in counts)  # 正样本对数
    total_neg_pairs = total_pairs - total_pos_pairs  # 负样本对数
    
    print(f"总对数: {total_pairs}, 正样本对数: {total_pos_pairs}, 负样本对数: {total_neg_pairs}")
    
    # 计算需要维护的 top-k 大小
    max_far = max(target_fars)
    top_k = max(int(total_neg_pairs * max_far), 1000)  # 至少保留1000个
    print(f"维护 top-{top_k} 负样本分数")
    
    # 使用最小堆维护 top-k 负样本分数
    neg_heap = []
    pos_scores = []
    
    block_size = 2048*5
    start = time.time()
    
    for i in tqdm(range(0, N, block_size), desc='Processing blocks'):
        block1 = feats_t[i:i+block_size]
        ids1 = ids_t[i:i+block_size]

        for j in range(i, N, block_size):
            block2 = feats_t[j:j+block_size]
            ids2 = ids_t[j:j+block_size]

            # 相似度计算
            sim_block = block1 @ block2.T

            # 标签匹配
            label_eq = (ids1[:, None] == ids2[None, :])

            # 上三角 mask
            if i == j:
                mask = torch.triu(torch.ones_like(sim_block, dtype=torch.bool), diagonal=1)
            else:
                mask = torch.ones_like(sim_block, dtype=torch.bool)

            # 提取有效的相似度和标签
            flat_sim = sim_block[mask]
            flat_labels = label_eq[mask]

            # 正样本分数直接收集
            pos_sim = flat_sim[flat_labels]
            if len(pos_sim) > 0:
                pos_scores.append(pos_sim.half().cpu())

            # 负样本分数只保留 top-k
            neg_sim = flat_sim[~flat_labels]
            k_local = min(top_k * 2, len(neg_sim))  # 取稍多一点，避免遗漏
            topk_neg = neg_sim.topk(k_local).values
            neg_candidates = topk_neg.half().cpu().numpy()  # 传到 CPU
            
            if len(neg_heap) == 0:
                neg_heap = neg_candidates
            else:
                neg_heap = np.concatenate([neg_heap, neg_candidates])
            
            if len(neg_heap) > top_k:
                # O(n) 分区操作
                neg_heap = np.partition(neg_heap, -top_k)[-top_k:]
                # 可选：排序便于后续判断最小值
                neg_heap = np.sort(neg_heap)  # 升序，最小值在 [0]
            else:
                neg_heap = np.sort(neg_heap)  # 维持有序

    print(f"计算矩阵耗时: {time.time() - start:.2f} 秒")
    
    # 转换正样本分数
    start = time.time()
    pos_scores = torch.cat(pos_scores).numpy() if pos_scores else np.array([])
    print(f"正样本处理耗时: {time.time() - start:.2f} 秒")
    print(f"正样本对数量: {len(pos_scores)}, 维护的负样本对数量: {len(neg_heap)}")

    # 计算 TPIR
    start = time.time()
    result, thresholds = compute_tpir_from_heap(neg_heap, pos_scores, total_neg_pairs, target_fars)
    print(f"计算 TPIR 耗时: {time.time() - start:.2f} 秒")
    
    return result, thresholds

In [26]:
with open('/root/zhaokj/work_test_v2/embeddings/webface12m.pkl', 'rb') as f:
    data = pickle.load(f)
embeddings = (data['collection']['features']).numpy()
img_input_feats = embeddings.copy()
img_input_feats = img_input_feats / np.sqrt(np.sum(img_input_feats ** 2, -1, keepdims=True))
target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]
result, thresholds = compute_tpir_optimized(img_input_feats, data['collection']['labels'].numpy(), target_fars=target_fars)

for far in target_fars:
    print(f"threshold_{far}: {thresholds[far]:.6f}, tpir_{far}: {result[f'tpir_at_far_{far}']:.4f}%")

总对数: 124711916176, 正样本对数: 28818474, 负样本对数: 124683097702
维护 top-1246830 负样本分数


Processing blocks:   0%|          | 0/49 [00:00<?, ?it/s]

计算矩阵耗时: 110.08 秒
正样本处理耗时: 0.01 秒
正样本对数量: 28818474, 维护的负样本对数量: 1246830
计算 TPIR 耗时: 2.05 秒
threshold_1e-05: 0.418945, tpir_1e-05: 77.0195%
threshold_1e-06: 0.476318, tpir_1e-06: 63.2694%
threshold_5e-07: 0.492920, tpir_5e-07: 58.7805%
threshold_1e-07: 0.532227, tpir_1e-07: 47.6835%
threshold_1e-08: 0.594727, tpir_1e-08: 30.1078%
threshold_1e-09: 0.696777, tpir_1e-09: 8.8982%
threshold_1e-10: 0.789551, tpir_1e-10: 1.4650%


In [52]:
def get_pos_neg_similarities(query_feats_list, query_ids, row_range, gpu_id, top_k, N, block_size=2048*5):
    """单个GPU上的计算函数 - 每个GPU负责特定的行范围"""
    device = f'cuda:{gpu_id}'
    feats_t = torch.tensor(query_feats_list, dtype=torch.float32).to(device)
    ids_t = torch.tensor(query_ids).to(device)
    
    neg_heap = []
    pos_scores = []
    
    row_start = row_range[0]
    row_end = row_range[1]

    # 外层循环: 只遍历当前GPU负责的行范围
    for i in tqdm(range(row_start, row_end, block_size), desc=f'GPU {gpu_id}', position=gpu_id, disable=True):
        i_end = min(i + block_size, row_end)
        block1 = feats_t[i:i_end]
        ids1 = ids_t[i:i_end]
        
        # 内层循环: 从当前行开始遍历所有列(保证上三角)
        for j in range(i, N, block_size):
            j_end = min(j + block_size, N)
            
            block2 = feats_t[j:j_end]
            ids2 = ids_t[j:j_end]

            # 相似度计算
            sim_block = block1 @ block2.T
            label_eq = (ids1[:, None] == ids2[None, :])

            # 上三角 mask
            if i == j:
                # 对角线块: 只取上三角(不含对角线)
                mask = torch.triu(torch.ones_like(sim_block, dtype=torch.bool), diagonal=1)
            else:
                # i < j: 完全在上三角
                mask = torch.ones_like(sim_block, dtype=torch.bool)

            flat_sim = sim_block[mask]
            flat_labels = label_eq[mask]

            # 正样本分数
            pos_sim = flat_sim[flat_labels]
            if len(pos_sim) > 0:
                pos_scores.append(pos_sim.half().cpu())

            # 负样本分数
            neg_sim = flat_sim[~flat_labels]
            if len(neg_sim) > 0:
                k_local = min(top_k * 2, len(neg_sim))
                topk_neg = neg_sim.topk(k_local).values
                neg_candidates = topk_neg.half().cpu().numpy()
                
                if len(neg_heap) == 0:
                    neg_heap = neg_candidates
                else:
                    neg_heap = np.concatenate([neg_heap, neg_candidates])
                
                if len(neg_heap) > top_k:
                    neg_heap = np.partition(neg_heap, -top_k)[-top_k:]
                    neg_heap = np.sort(neg_heap)
                else:
                    neg_heap = np.sort(neg_heap)
    
    pos_scores = torch.cat(pos_scores).numpy() if pos_scores else np.array([])
    
    return pos_scores, neg_heap


def compute_tpir_optimized2(query_feats_list, query_ids, target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]):
    """
    优化版本:只维护 top 1e-5 的负样本分数, 使用joblib多GPU并行计算
    """
    N = len(query_ids)
    total_pairs = N * (N - 1) // 2
    unique_ids, counts = np.unique(query_ids, return_counts=True)
    total_pos_pairs = sum(c * (c - 1) // 2 for c in counts)
    total_neg_pairs = total_pairs - total_pos_pairs
    print(f"总对数: {total_pairs}, 正样本对数: {total_pos_pairs}, 负样本对数: {total_neg_pairs}")
    
    max_far = max(target_fars)
    top_k = max(int(total_neg_pairs * max_far), 1000)
    print(f"维护 top-{top_k} 负样本分数")
    
    # 按行范围分配给4个GPU
    num_gpus = 8
    row_ranges = [(i * (N // num_gpus), (i + 1) * (N // num_gpus)) for i in range(num_gpus)]
    row_ranges[-1] = (row_ranges[-1][0], N)
    print("行范围分配:", row_ranges)
    start = time.time()
    results = Parallel(n_jobs=num_gpus, backend='threading')(
        delayed(get_pos_neg_similarities)(
            query_feats_list, 
            query_ids, 
            row_ranges[gpu_id], 
            gpu_id, 
            top_k, 
            N
        )
        for gpu_id in range(num_gpus)
    )
    print(f"计算矩阵耗时: {time.time() - start:.2f} 秒")
    
    get_reusable_executor().shutdown(wait=True)
    all_pos_scores = []
    all_neg_heap = []
    
    for pos, neg in results:
        if len(pos) > 0:
            all_pos_scores.append(pos)
        if len(neg) > 0:
            all_neg_heap.append(neg)
    
    pos_scores = np.concatenate(all_pos_scores) if all_pos_scores else np.array([])
    neg_heap = np.concatenate(all_neg_heap) if all_neg_heap else np.array([])
    if len(neg_heap) > top_k:
        neg_heap = np.partition(neg_heap, -top_k)[-top_k:]
        neg_heap = np.sort(neg_heap)
    
    print(f"正样本对数量: {len(pos_scores)}, 维护的负样本对数量: {len(neg_heap)}")

    start = time.time()
    result, thresholds = compute_tpir_from_heap(neg_heap, pos_scores, total_neg_pairs, target_fars)
    print(f"计算 TPIR 耗时: {time.time() - start:.2f} 秒")
    
    return result, thresholds

In [53]:
target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]
result, thresholds = compute_tpir_optimized2(img_input_feats, data['collection']['labels'].numpy(), target_fars=target_fars)

for far in target_fars:
    print(f"threshold_{far}: {thresholds[far]:.6f}, tpir_{far}: {result[f'tpir_at_far_{far}']:.4f}%")

总对数: 124711916176, 正样本对数: 28818474, 负样本对数: 124683097702
维护 top-1246830 负样本分数
行范围分配: [(0, 62428), (62428, 124856), (124856, 187284), (187284, 249712), (249712, 312140), (312140, 374568), (374568, 436996), (436996, 499424)]
计算矩阵耗时: 37.67 秒
正样本对数量: 28818474, 维护的负样本对数量: 1246830
计算 TPIR 耗时: 2.77 秒
threshold_1e-05: 0.418945, tpir_1e-05: 77.0195%
threshold_1e-06: 0.476318, tpir_1e-06: 63.2694%
threshold_5e-07: 0.492920, tpir_5e-07: 58.7805%
threshold_1e-07: 0.532227, tpir_1e-07: 47.6835%
threshold_1e-08: 0.594727, tpir_1e-08: 30.1078%
threshold_1e-09: 0.696777, tpir_1e-09: 8.8982%
threshold_1e-10: 0.789551, tpir_1e-10: 1.4650%


In [54]:
with open('/root/zhaokj/work_test_v2/embeddings/epoch48.pkl', 'rb') as f:
    data = pickle.load(f)
embeddings = (data['collection']['features']).numpy()
img_input_feats = embeddings.copy()
img_input_feats = img_input_feats / np.sqrt(np.sum(img_input_feats ** 2, -1, keepdims=True))
target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]
result, thresholds = compute_tpir_optimized2(img_input_feats, data['collection']['labels'].numpy(), target_fars=target_fars)

for far in target_fars:
    print(f"threshold_{far}: {thresholds[far]:.6f}, tpir_{far}: {result[f'tpir_at_far_{far}']:.4f}%")

总对数: 124711916176, 正样本对数: 28818474, 负样本对数: 124683097702
维护 top-1246830 负样本分数
行范围分配: [(0, 62428), (62428, 124856), (124856, 187284), (187284, 249712), (249712, 312140), (312140, 374568), (374568, 436996), (436996, 499424)]
计算矩阵耗时: 40.13 秒
正样本对数量: 28818474, 维护的负样本对数量: 1246830
计算 TPIR 耗时: 2.30 秒
threshold_1e-05: 0.287354, tpir_1e-05: 98.1305%
threshold_1e-06: 0.348145, tpir_1e-06: 95.0025%
threshold_5e-07: 0.365723, tpir_5e-07: 93.5541%
threshold_1e-07: 0.405273, tpir_1e-07: 89.1780%
threshold_1e-08: 0.458008, tpir_1e-08: 80.7520%
threshold_1e-09: 0.504883, tpir_1e-09: 70.8476%
threshold_1e-10: 0.549316, tpir_1e-10: 59.5737%


In [55]:
with open('/root/zhaokj/work_test_v2/embeddings/s3_full_29.pkl', 'rb') as f:
    data = pickle.load(f)
embeddings = (data['collection']['features']).numpy()
img_input_feats = embeddings.copy()
img_input_feats = img_input_feats / np.sqrt(np.sum(img_input_feats ** 2, -1, keepdims=True))
target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]
result, thresholds = compute_tpir_optimized2(img_input_feats, data['collection']['labels'].numpy(), target_fars=target_fars)

for far in target_fars:
    print(f"threshold_{far}: {thresholds[far]:.6f}, tpir_{far}: {result[f'tpir_at_far_{far}']:.4f}%")

总对数: 124711916176, 正样本对数: 28818474, 负样本对数: 124683097702
维护 top-1246830 负样本分数
行范围分配: [(0, 62428), (62428, 124856), (124856, 187284), (187284, 249712), (249712, 312140), (312140, 374568), (374568, 436996), (436996, 499424)]
计算矩阵耗时: 40.26 秒
正样本对数量: 28818474, 维护的负样本对数量: 1246830
计算 TPIR 耗时: 2.40 秒
threshold_1e-05: 0.294678, tpir_1e-05: 97.7626%
threshold_1e-06: 0.350098, tpir_1e-06: 94.6997%
threshold_5e-07: 0.365967, tpir_5e-07: 93.3538%
threshold_1e-07: 0.400879, tpir_1e-07: 89.4405%
threshold_1e-08: 0.445801, tpir_1e-08: 82.2079%
threshold_1e-09: 0.482178, tpir_1e-09: 74.4327%
threshold_1e-10: 0.506348, tpir_1e-10: 68.4288%


In [56]:
with open('/data1/dataset/rec/test2_rec/done_list.txt', 'r') as f:
    lines = f.read().splitlines()
name_idx_dict = {}
for idx, line in tqdm(enumerate(lines), total=len(lines)):
    image_name = line.split()[1].split('/')[1]
    name_idx_dict[image_name] = idx


  0%|          | 0/499424 [00:00<?, ?it/s]

In [63]:
with open('/root/zhaokj/CVLface/cvlface/research/recognition/code/work_0925/001_0928_image_list.txt', 'r') as f:
    lines = f.read().splitlines()
indexs = []

for line in lines:
    image_split = line.split('+')
    image_name = f'{image_split[1]}+{image_split[2]}'
    indexs.append(name_idx_dict[image_name])
    

In [69]:
indexs = sorted(indexs)
indexs = np.array(indexs)

In [70]:
with open('/root/zhaokj/work_test_v2/embeddings/webface12m.pkl', 'rb') as f:
    data = pickle.load(f)
embeddings = (data['collection']['features']).numpy()
img_input_feats = embeddings.copy()
img_input_feats = img_input_feats / np.sqrt(np.sum(img_input_feats ** 2, -1, keepdims=True))

img_input_feats = img_input_feats[indexs]
query_ids = data['collection']['labels'].numpy()[indexs]
target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]
result, thresholds = compute_tpir_optimized2(img_input_feats, query_ids, target_fars=target_fars)

for far in target_fars:
    print(f"threshold_{far}: {thresholds[far]:.6f}, tpir_{far}: {result[f'tpir_at_far_{far}']:.4f}%")

总对数: 122145732411, 正样本对数: 28147118, 负样本对数: 122117585293
维护 top-1221175 负样本分数
行范围分配: [(0, 61782), (61782, 123564), (123564, 185346), (185346, 247128), (247128, 308910), (308910, 370692), (370692, 432474), (432474, 494259)]
计算矩阵耗时: 35.75 秒
正样本对数量: 28147118, 维护的负样本对数量: 1221175
计算 TPIR 耗时: 2.66 秒
threshold_1e-05: 0.418945, tpir_1e-05: 77.6854%
threshold_1e-06: 0.475830, tpir_1e-06: 64.1420%
threshold_5e-07: 0.492676, tpir_5e-07: 59.5929%
threshold_1e-07: 0.531250, tpir_1e-07: 48.6727%
threshold_1e-08: 0.590820, tpir_1e-08: 31.7001%
threshold_1e-09: 0.684082, tpir_1e-09: 10.9975%
threshold_1e-10: 0.784668, tpir_1e-10: 1.6708%


In [73]:
with open('/root/zhaokj/work_test_v2/embeddings/epoch48.pkl', 'rb') as f:
    data = pickle.load(f)
embeddings = (data['collection']['features']).numpy()
img_input_feats = embeddings.copy()
img_input_feats = img_input_feats / np.sqrt(np.sum(img_input_feats ** 2, -1, keepdims=True))

img_input_feats = img_input_feats[indexs]
query_ids = data['collection']['labels'].numpy()[indexs]

target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]
result, thresholds = compute_tpir_optimized2(img_input_feats, query_ids, target_fars=target_fars)

for far in target_fars:
    print(f"threshold_{far}: {thresholds[far]:.6f}, tpir_{far}: {result[f'tpir_at_far_{far}']:.4f}%")

总对数: 122145732411, 正样本对数: 28147118, 负样本对数: 122117585293
维护 top-1221175 负样本分数
行范围分配: [(0, 61782), (61782, 123564), (123564, 185346), (185346, 247128), (247128, 308910), (308910, 370692), (370692, 432474), (432474, 494259)]
计算矩阵耗时: 41.02 秒
正样本对数量: 28147118, 维护的负样本对数量: 1221175
计算 TPIR 耗时: 2.70 秒
threshold_1e-05: 0.287109, tpir_1e-05: 98.2092%
threshold_1e-06: 0.347656, tpir_1e-06: 95.2137%
threshold_5e-07: 0.364990, tpir_5e-07: 93.8365%
threshold_1e-07: 0.404053, tpir_1e-07: 89.6619%
threshold_1e-08: 0.456299, tpir_1e-08: 81.5509%
threshold_1e-09: 0.503418, tpir_1e-09: 71.7769%
threshold_1e-10: 0.545898, tpir_1e-10: 61.1194%


In [ ]:
with open('/root/zhaokj/work_test_v2/embeddings/s3_full_29.pkl', 'rb') as f:
    data = pickle.load(f)
embeddings = (data['collection']['features']).numpy()
img_input_feats = embeddings.copy()
img_input_feats = img_input_feats / np.sqrt(np.sum(img_input_feats ** 2, -1, keepdims=True))
img_input_feats = img_input_feats[indexs]
query_ids = data['collection']['labels'].numpy()[indexs]

target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]
result, thresholds = compute_tpir_optimized2(img_input_feats, query_ids, target_fars=target_fars)

for far in target_fars:
    print(f"threshold_{far}: {thresholds[far]:.6f}, tpir_{far}: {result[f'tpir_at_far_{far}']:.4f}%")

总对数: 122145732411, 正样本对数: 28147118, 负样本对数: 122117585293
维护 top-1221175 负样本分数
行范围分配: [(0, 61782), (61782, 123564), (123564, 185346), (185346, 247128), (247128, 308910), (308910, 370692), (370692, 432474), (432474, 494259)]
计算矩阵耗时: 37.20 秒
正样本对数量: 28147118, 维护的负样本对数量: 1221175
计算 TPIR 耗时: 2.21 秒
threshold_1e-05: 0.294434, tpir_1e-05: 97.8906%
threshold_1e-06: 0.349609, tpir_1e-06: 94.9635%
threshold_5e-07: 0.365234, tpir_5e-07: 93.6860%
threshold_1e-07: 0.400391, tpir_1e-07: 89.8670%
threshold_1e-08: 0.445312, tpir_1e-08: 82.8023%
threshold_1e-09: 0.480957, tpir_1e-09: 75.3200%
threshold_1e-10: 0.506348, tpir_1e-10: 69.0794%


: 